# Tono · atacar el fallo en melanoma

> ⚠️ **NO ES UNA HERRAMIENTA DIAGNÓSTICA.** Proyecto educativo y experimental.

El modelo actual se deja **1 de cada 6 melanomas** (16%), frente al 7% en
carcinomas. Falla más en lo más letal. Y dos fotos reales que no detectaba
señalaron una causa concreta: **el preprocesado destruye las lesiones
pequeñas**.

    lesión que ocupa el 3% del encuadre, recorte central -> 0,054  (no detecta)
    la misma con multi-recorte                           -> 0,277  (detecta)

## Tres cambios, medidos por separado

| | Cambio | Por qué |
|---|---|---|
| **1** | 320 px en vez de 224 | el melanoma se distingue por detalles finos que a 224 se pierden |
| **2** | Fitzpatrick17k **+** PAD-UFES-20 | entrenar con una sola fuente hundió al modelo anterior al cambiar de dominio (0,899 → 0,684) |
| **3** | Inferencia por multi-recorte | recorre la imagen por trozos y toma el máximo, en vez de mirar solo el centro |

## La métrica que decide

**No es el AUROC: es la sensibilidad en melanoma.** Un modelo que suba el AUROC
global mientras sigue perdiendo melanomas no sirve para nada aquí. Y como el
multi-recorte sube *todas* las puntuaciones, el umbral se recalibra en
validación antes de comparar nada.

In [ ]:
import subprocess, sys, os, time, glob, json
T0 = time.time()

for nombre, url in [('tono', 'https://github.com/GGGuardin/tono.git'),
                    ('cxr', 'https://github.com/GGGuardin/chest-xray-pneumonia.git')]:
    subprocess.run(['rm', '-rf', '/tmp/' + nombre], check=False)
    subprocess.run(['git', 'clone', '--depth', '1', '-q', url, '/tmp/' + nombre], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations'], check=True)

import torch
cap = torch.cuda.get_device_capability(0)
assert 'sm_%d%d' % cap in torch.cuda.get_arch_list(), 'GPU no soportada, relanza con T4'
torch.zeros(8, device='cuda').sum().item()
print('GPU:', torch.cuda.get_device_name(0), '| CUDA OK')

## 1. Manifiesto combinado

In [ ]:
OUT = '/kaggle/working'
os.chdir('/tmp/tono')

anclas = glob.glob('/kaggle/input/**/fitzpatrick17k*.csv', recursive=True)
FITZ = os.path.dirname(anclas[0])
pad = [p for p in glob.glob('/kaggle/input/**/*.csv', recursive=True)
       if 'fitzpatrick' not in os.path.basename(p).lower()]
PAD = os.path.dirname(pad[0])
while PAD != '/kaggle/input' and not glob.glob(os.path.join(PAD, '**', '*.png'), recursive=True):
    PAD = os.path.dirname(PAD)
print('FITZ:', FITZ, '| PAD:', PAD)

!python -m datos.fitzpatrick17k --root {FITZ} --out {OUT}/m_fitz.csv
!python -m datos.pad_ufes --root {PAD} --out {OUT}/m_pad.csv
!python -m scripts.combinar_manifiestos --entradas {OUT}/m_fitz.csv {OUT}/m_pad.csv --out {OUT}/manifiesto_combinado.csv

## 2. Entrenamiento a 320 px con las dos fuentes

In [ ]:
os.chdir('/tmp/cxr')
!python -m src.train --config /tmp/tono/configs/combinado_320.yaml --manifest {OUT}/manifiesto_combinado.csv --out-dir {OUT}/runs/combinado

In [ ]:
import pandas as pd
h = pd.read_csv(OUT + '/runs/combinado/history.csv')
print(h.to_string(index=False))

## 3. Cuatro evaluaciones para separar los efectos

Modelo nuevo con recorte central y con multi-recorte, sobre validación (para
fijar umbrales) y sobre test (para medir).

In [ ]:
CKPT = OUT + '/runs/combinado/best.pth'
M = OUT + '/manifiesto_combinado.csv'

for split in ['val', 'test']:
    !python -m src.evaluate --checkpoint {CKPT} --manifest {M} --split {split} --out-dir {OUT}/reports/centro_{split} --n-boot 300

In [ ]:
os.chdir('/tmp/tono')
for split in ['val', 'test']:
    !python scripts/evaluar_multirecorte.py --checkpoint {CKPT} --manifest {M} --split {split} --out {OUT}/reports/multi_{split}/predictions.csv --escalas 1.0,0.6,0.4 --paso 0.5
os.chdir('/tmp/cxr')

## 4. Comparación, con la sensibilidad en melanoma al frente

Cada estrategia se evalúa con **su propio umbral**, derivado de validación
fijando una sensibilidad objetivo del 90%. Comparar con umbrales distintos sería
el único modo justo: el multi-recorte desplaza toda la escala.

In [ ]:
import numpy as np
sys.path.insert(0, '/tmp/cxr')
from src.metrics import binary_metrics
from sklearn.metrics import roc_auc_score, roc_curve

def umbral_sens(y, p, objetivo=0.90):
    _, tpr, u = roc_curve(y, p)
    ok = np.where(tpr >= objetivo)[0]
    return float(u[ok[0]]) if len(ok) else 0.5

# El manifiesto combinado pierde la columna de diagnostico fino: se recupera del
# manifiesto de Fitzpatrick para poder aislar los melanomas.
fz = pd.read_csv(OUT + '/m_fitz.csv')[['image_path', 'view']]
fz.columns = ['image_path', 'categoria']

resultados = {}
for nombre in ['centro', 'multi']:
    val = pd.read_csv(OUT + '/reports/%s_val/predictions.csv' % nombre)
    test = pd.read_csv(OUT + '/reports/%s_test/predictions.csv' % nombre)
    u = umbral_sens(val.label.values, val.prob.values, 0.90)
    m = binary_metrics(test.label.values, test.prob.values, u)

    t = test.merge(fz, on='image_path', how='left')
    mel = t[(t.label == 1) & (t.categoria == 'malignant melanoma')]
    otras = t[(t.label == 1) & (t.categoria.notna()) & (t.categoria != 'malignant melanoma')]
    resultados[nombre] = {
        'umbral': round(u, 4),
        'auroc': round(m['auroc'], 4),
        'sensibilidad_global': round(m['sensibilidad'], 4),
        'especificidad': round(m['especificidad'], 4),
        'melanomas_n': int(len(mel)),
        'sensibilidad_melanoma': round(float((mel.prob >= u).mean()), 4) if len(mel) else None,
        'sensibilidad_otras_malignas': round(float((otras.prob >= u).mean()), 4) if len(otras) else None,
    }

print('%-24s %-10s %-10s %-12s %s' % ('', 'AUROC', 'sens.glob', 'especif.', 'SENS. MELANOMA'))
for n, r in resultados.items():
    print('%-24s %-10.4f %-10.4f %-12.4f %s (n=%d)'
          % (n, r['auroc'], r['sensibilidad_global'], r['especificidad'],
             r['sensibilidad_melanoma'], r['melanomas_n']))
print()
print('Referencia del modelo anterior (224 px, solo Fitzpatrick17k):')
print('  sensibilidad en melanoma 0.84  ->  se escapaba 1 de cada 6')

In [ ]:
# Por dominio: un promedio entre atlas y fotos de movil esconde si uno va mal
for nombre in ['centro', 'multi']:
    test = pd.read_csv(OUT + '/reports/%s_test/predictions.csv' % nombre)
    u = resultados[nombre]['umbral']
    print('---', nombre, '---')
    for fuente, parte in test.groupby('source'):
        if parte.label.nunique() < 2:
            continue
        m = binary_metrics(parte.label.values, parte.prob.values, u)
        print('  %-18s n=%-5d AUROC %.4f  sens %.4f' % (fuente, m['n'], m['auroc'], m['sensibilidad']))
    resultados[nombre]['por_dominio'] = {
        str(f): round(binary_metrics(p.label.values, p.prob.values, u)['auroc'], 4)
        for f, p in test.groupby('source') if p.label.nunique() > 1}

In [ ]:
!python -m src.fairness --predictions {OUT}/reports/multi_test/predictions.csv --out-dir {OUT}/reports/equidad --attributes fitzpatrick,source --intersect "" --min-n 25

In [ ]:
mejor = max(resultados.items(),
            key=lambda kv: kv[1]['sensibilidad_melanoma'] or 0)
resumen = {
    'objetivo': 'reducir los melanomas no detectados (antes: 1 de cada 6)',
    'cambios': ['320 px', 'Fitzpatrick17k + PAD-UFES-20', 'inferencia multi-recorte'],
    'metrica_que_decide': 'sensibilidad en melanoma, no AUROC',
    'umbral_derivado_en': 'validacion, fijando sensibilidad 0.90',
    'resultados': resultados,
    'mejor': mejor[0],
    'minutos': round((time.time() - T0) / 60, 1),
}
json.dump(resumen, open(OUT + '/resumen.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(resumen, indent=2, ensure_ascii=False))

import shutil
for f_ in glob.glob(OUT + '/m_*.csv') + glob.glob(OUT + '/manifiesto_*.csv'):
    shutil.move(f_, '/tmp/' + os.path.basename(f_))